# Action Castle — Simultaneous Turns

This is a companion to **`multi_agent_action_castle.ipynb`**, which builds the full
Action Castle game in the engine's default **sequential** turn mode (the player acts,
then each NPC observes the *already-changed* world and acts, one after another).

This notebook focuses on the engine's opt-in **simultaneous** turn mode (issue #25):
`Game(..., turn_mode="simultaneous")`. Like its companion it runs **fully offline** and
needs no API key — the NPCs here are driven by tiny `ScriptedAgent` rules.

> **Run this from the `notebooks/` directory** (the Jupyter kernel's working directory
> must contain `hw1_solution/`), with the repo installed: `pip install -e ".[dev]"`.


In [1]:
from rich.console import Console

from text_adventure_games import games, things
from text_adventure_games.npc import ScriptedAgent
from text_adventure_games.reporting import NORMAL, RichTerminalRenderer

# Custom actions reused from the HW1 solution.
from hw1_solution.action_castle import Growl, Warn


def force_rich_notebook_output(game, level=NORMAL):
    """Use Rich's Jupyter renderer for parser output in this notebook."""
    game.parser.set_renderer(
        RichTerminalRenderer(
            level=level,
            console=Console(force_jupyter=True, width=100),
        )
    )
    return game


def build_player():
    """The hero: a simple peasant with a lamp."""
    player = things.Character(
        name="The player",
        description="You are a simple peasant destined for greatness.",
        persona="I am on an adventure.",
    )
    lamp = things.Item("lamp", "a lamp", "A LAMP.")
    lamp.set_property("is_lightable", True)
    lamp.add_command_hint("light lamp")
    player.add_to_inventory(lamp)
    return player


## How a simultaneous round works

In **sequential** mode each NPC observes the *already-changed* world and acts, one
after another. The opt-in **simultaneous** mode — `Game(..., turn_mode="simultaneous")`
— runs each round in two phases instead:

1. **gather** — every NPC agent picks a command against the *same turn-start snapshot*,
   **before** the player's command lands. Nobody sees what anyone else does this round.
2. **resolve** — the player's command resolves first, then the NPCs' in `initiative`
   order (ties keep gather order). Same-turn **contention** is settled here: the loser's
   gathered command fails the precondition gate, the failure reason is fed back to its
   agent for one retry, and an unrecovered failure is recorded as an `action_failed`
   event.

Two wiring details: agents attach with `character.set_agent(agent)` — the loop calls
`decide()` itself, so there's no behavior closure — and `initiative` is just a property
on the character.

The scene: one fish on the drawbridge, and everyone wants it. The troll is *gathered*
first, but the guard is *quicker* (initiative 3 vs 1). Each brain is a plain
`ScriptedAgent` rule — grab the fish if you can see it; if your grab fails, the troll
vents with a growl while the guard just gives up:


In [2]:
def grabby(fallback=None):
    """Grab the fish on sight; on a failed grab, fall back (None = give up)."""

    def rule(observation):
        if "Your previous command" in observation:  # the resolve-time reflection
            return fallback
        if "* fish" in observation:
            return "take fish"
        return None

    return rule


def build_standoff():
    """A one-room scene: two hungry NPCs, one fish, simultaneous turns."""
    bridge = things.Location("Drawbridge", "A weathered drawbridge across the moat.")
    fish = things.Item("fish", "a plump raw fish", "IT SMELLS DELICIOUS, IF YOU ARE A TROLL.")
    bridge.add_item(fish)

    player = build_player()
    troll = things.Character("troll", "A mean troll", "I am hungry. That fish is mine.")
    guard = things.Character("guard", "A castle guard", "I skipped lunch. That fish is mine.")
    troll.set_property("initiative", 1)
    guard.set_property("initiative", 3)  # quicker on the draw

    game = games.Game(
        bridge,
        player,
        characters=[troll, guard],
        custom_actions=[Growl, Warn],
        turn_mode="simultaneous",  # <-- the new mode
    )
    bridge.add_character(troll)
    bridge.add_character(guard)
    force_rich_notebook_output(game)

    troll.set_agent(ScriptedAgent(grabby(fallback="growl")))
    guard.set_agent(ScriptedAgent(grabby(fallback=None)))
    return game

In [3]:
game = build_standoff()
game.parser.echo_commands = True
game.do_command("look")   # the player just watches

print()
print("troll has:", list(game.characters["troll"].inventory) or "nothing")
print("guard has:", list(game.characters["guard"].inventory) or "nothing")

> look

» DRAWBRIDGE
A weathered drawbridge across the moat.

You see:
 * fish - a plump raw fish
Characters:
 * troll - A mean troll
 * guard - A castle guard

Turn 1 ─────────────────────────────────────────────────────────────────────────────────────────────

guard

  · act     take fish

...

  reflect  I don't see it.

  · act     growl

> growl

» Troll growls menacingly at The player.


troll has: nothing
guard has: ['fish']


Both NPCs decided to `take fish` from the **same snapshot** — the troll never saw the
guard move first. At resolve time the guard's higher initiative won the race; the
troll's gathered command hit the precondition gate ("I don't see it."), the failure was
fed back as a reflection, and its one retry growled instead. Because the retry
*recovered*, nothing was logged as a failure.

Now the player joins the scramble. The player **always resolves first** (the design-doc
rule — initiative only orders the NPCs), so both NPCs lose: each decided to grab a fish
that, by the time it moved, was already in our pocket. That is the snapshot at work —
in sequential mode they would simply have *seen* the empty plank and never tried. The
troll recovers with its growl again; the guard gives up, and its unrecovered failure is
recorded in the event log as an `action_failed` event — the conflict's paper trail.

In [4]:
game = build_standoff()
game.parser.echo_commands = True
game.do_command("take fish")   # the player wants lunch too

print()
print("The conflict's paper trail:")
for event in game.events:
    print(f"  turn {event.turn:>2}  {event.actor:<12} {event.action:<14} {event.summary}")

> take fish

» The player got the fish.

Turn 1 ─────────────────────────────────────────────────────────────────────────────────────────────

guard

  · act     take fish

...

  reflect  I don't see it.

  · act     growl

> growl

» Troll growls menacingly at The player.


The conflict's paper trail:
  turn  0  The player   get            take fish
  turn  1  guard        action_failed  I don't see it.
  turn  1  troll        growl          growl
